# AgriTwin ML Pipeline: 05 - MobileNetV2 Transfer Learning

## 1. Pretrained MobileNetV2 Architecture
Using ImageNet pretrained backbone for fast convergence and lightweight inference.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

inputs = layers.Input(shape=(224, 224, 3))
x = preprocess_input(inputs)

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation="softmax")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="AgriTwin_MobileNetV2")
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 2. Two-Phase Training Strategy
- Phase A: Feature Extraction (Backbone Frozen)
- Phase B: Fine-Tuning Top 40 Inverted Residual Layers with lr=1e-5

In [ ]:
# Phase A
# history_a = model.fit(train_ds, validation_data=val_ds, epochs=8)

# Phase B: Unfreeze upper layers
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
# history_b = model.fit(train_ds, validation_data=val_ds, epochs=5)